In [1]:
# ============================================
# 0. 설치
# ============================================
!pip -q install transformers accelerate datasets tqdm

import os
import gc
import zipfile
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import files

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [2]:
# ============================================
# 1. 새 전처리 파일 업로드
# ============================================
uploaded = files.upload()

# 파일명이 modified_preprocessing.csv인지 확인
print(os.listdir())

Saving modified_preprocessing.csv to modified_preprocessing.csv
['.config', 'modified_preprocessing.csv', 'sample_data']


In [3]:
# ============================================
# 2. 데이터 불러오기
# ============================================
DATA_PATH = "modified_preprocessing.csv"

df = pd.read_csv(DATA_PATH)

df["created_time"] = pd.to_datetime(df["created_time"], errors="coerce")
df["Date"] = df["created_time"].dt.date

df["cleaned_text"] = df["cleaned_text"].fillna("").astype(str)
df = df[df["cleaned_text"].str.strip() != ""].copy()

# 너무 긴 댓글은 모델 입력 제한 때문에 적당히 자르기
df["text_for_model"] = df["cleaned_text"].str.slice(0, 1000)

print(df.shape)
display(df[["Date", "cleaned_text"]].head())
print(df["Date"].min(), df["Date"].max())

(55473, 27)


,Date,cleaned_text
0,2023-11-07,Palestinians are so anti-jewish that hamas' pu...
1,2023-11-07,"War is bad,hamas started this and then ran to ..."
2,2023-11-07,I don’t deny that Hamas is also pursuing a pol...
3,2023-11-07,"Yes, why would they put their soldiers in harm..."
4,2023-11-07,"Israel is not allowed to root out Hamas, they ..."


2023-09-07 2023-11-07


In [4]:
# ============================================
# 3. 사용할 모델 목록
# ============================================
MODEL_CONFIGS = [
    {
        "name": "bert_goemotions",
        "model_id": "monologg/bert-base-cased-goemotions-original",
        "type": "goemotions"
    },
    {
        "name": "distilbert_goemotions",
        "model_id": "joeddav/distilbert-base-uncased-go-emotions-student",
        "type": "goemotions"
    },
    {
        "name": "distilroberta_goemotions",
        "model_id": "SamLowe/roberta-base-go_emotions",
        "type": "goemotions"
    },
    {
        "name": "modernbert_goemotions",
        "model_id": "answerdotai/ModernBERT-base",
        "type": "goemotions"
    },
    {
        "name": "twitter_roberta",
        "model_id": "cardiffnlp/twitter-roberta-base-sentiment-latest",
        "type": "sentiment"
    }
]

In [5]:
# ============================================
# 4. 감정분석 함수
# ============================================
def run_model_daily(df, model_name, model_id, model_type, batch_size=32):
    print("=" * 80)
    print(f"Running model: {model_name}")
    print(f"Model ID: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id)
    model.to(device)
    model.eval()

    id2label = model.config.id2label
    labels = [id2label[i] for i in range(len(id2label))]
    labels = [str(x).lower() for x in labels]

    print("Labels:", labels)

    texts = df["text_for_model"].tolist()
    dates = df["Date"].tolist()

    all_scores = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits

        # GoEmotions는 보통 multi-label이라 sigmoid
        # Twitter sentiment는 positive/neutral/negative라 softmax
        if model_type == "goemotions":
            probs = torch.sigmoid(logits)
        else:
            probs = torch.softmax(logits, dim=-1)

        all_scores.append(probs.cpu().numpy())

    all_scores = np.vstack(all_scores)

    score_df = pd.DataFrame(all_scores, columns=labels)
    score_df["Date"] = dates

    # 날짜별 평균 감정 점수
    daily = score_df.groupby("Date")[labels].mean().reset_index()

    # 날짜별 댓글 수
    counts = df.groupby("Date").size().reset_index(name="comment_count")
    daily = daily.merge(counts, on="Date", how="left")

    # 저장
    output_name = f"{model_name}_daily.csv"
    daily.to_csv(output_name, index=False)

    print(f"Saved: {output_name}")
    display(daily.head())

    del tokenizer, model
    torch.cuda.empty_cache()
    gc.collect()

    return daily

In [6]:
# ============================================
# 4. 감정분석 함수
# ============================================
def run_model_daily(df, model_name, model_id, model_type, batch_size=32):
    print("=" * 80)
    print(f"Running model: {model_name}")
    print(f"Model ID: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id)
    model.to(device)
    model.eval()

    id2label = model.config.id2label
    labels = [id2label[i] for i in range(len(id2label))]
    labels = [str(x).lower() for x in labels]

    print("Labels:", labels)

    texts = df["text_for_model"].tolist()
    dates = df["Date"].tolist()

    all_scores = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits

        # GoEmotions는 보통 multi-label이라 sigmoid
        # Twitter sentiment는 positive/neutral/negative라 softmax
        if model_type == "goemotions":
            probs = torch.sigmoid(logits)
        else:
            probs = torch.softmax(logits, dim=-1)

        all_scores.append(probs.cpu().numpy())

    all_scores = np.vstack(all_scores)

    score_df = pd.DataFrame(all_scores, columns=labels)
    score_df["Date"] = dates

    # 날짜별 평균 감정 점수
    daily = score_df.groupby("Date")[labels].mean().reset_index()

    # 날짜별 댓글 수
    counts = df.groupby("Date").size().reset_index(name="comment_count")
    daily = daily.merge(counts, on="Date", how="left")

    # 저장
    output_name = f"{model_name}_daily.csv"
    daily.to_csv(output_name, index=False)

    print(f"Saved: {output_name}")
    display(daily.head())

    del tokenizer, model
    torch.cuda.empty_cache()
    gc.collect()

    return daily

In [7]:
# ============================================
# 5. 전체 모델 한 번에 실행
# ============================================
daily_results = {}

for config in MODEL_CONFIGS:
    try:
        daily = run_model_daily(
            df=df,
            model_name=config["name"],
            model_id=config["model_id"],
            model_type=config["type"],
            batch_size=32
        )
        daily_results[config["name"]] = daily

    except Exception as e:
        print("=" * 80)
        print(f"ERROR in {config['name']}")
        print(e)
        print("이 모델은 건너뛰고 다음 모델로 넘어감.")
        torch.cuda.empty_cache()
        gc.collect()

Running model: bert_goemotions
Model ID: monologg/bert-base-cased-goemotions-original


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.67k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

...

Labels: ['negative', 'neutral', 'positive']


  0%|          | 0/1734 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Saved: twitter_roberta_daily.csv


,Date,negative,neutral,positive,comment_count
0,2023-09-07,0.454486,0.396909,0.148604,91
1,2023-09-08,0.499195,0.393306,0.107498,40
2,2023-09-09,0.720283,0.244823,0.034894,19
3,2023-09-10,0.456667,0.387956,0.155377,52
4,2023-09-11,0.601146,0.307927,0.090927,239


In [8]:
# ============================================
# 6. 생성된 파일 확인
# ============================================
daily_files = [f for f in os.listdir() if f.endswith("_daily.csv")]
daily_files

['modernbert_goemotions_daily.csv',
 'distilroberta_goemotions_daily.csv',
 'bert_goemotions_daily.csv',
 'distilbert_goemotions_daily.csv',
 'twitter_roberta_daily.csv']

In [9]:
# ============================================
# 7. 결과 파일 zip으로 묶어서 다운로드
# ============================================
zip_name = "new_emotion_daily_results.zip"

with zipfile.ZipFile(zip_name, "w") as zipf:
    for f in daily_files:
        zipf.write(f)

files.download(zip_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>